# N08 · From initial perturbations to a galaxy map

## The first step of the forward model

**Module 3 — Multi-wavelength large-scale-structure cosmology (PhD onboarding).**
The thesis forward model starts by turning the linear power spectrum (N07) into a realised matter
field and then into a galaxy map. This notebook builds a Gaussian field with a JAX FFT, applies the
lognormal transform and galaxy bias, Poisson-samples galaxies, and recovers the input $P(k)$ — then
shows the map-making is a **differentiable** forward model. Finally it uses **GLASS** (Tessore et al.
2023) for a realistic on-sphere lognormal mock.

**Key tools:** JAX (`jax.numpy.fft`), `numpy`, and `glass`.


### Learning objectives

1. Generate a Gaussian random field from $P(k)$ with an FFT (in JAX).
2. Apply the lognormal transform to enforce $\delta>-1$ and reproduce the observed one-point statistics.
3. Add linear galaxy bias and Poisson-sample a galaxy field.
4. Measure $P(k)$ back from the mock and compare to the input.
5. Differentiate the map-making forward model and connect to field-level inference (N13).
6. Build a correlated lognormal mock on the sphere with GLASS.


## References

- Peebles (1980), *The Large-Scale Structure of the Universe* — Gaussian fields and $P(k)$.
- Coles & Jones (1991), MNRAS 248, 1 — the lognormal density field
  [doi:10.1093/mnras/248.1.1](https://doi.org/10.1093/mnras/248.1.1).
- Xavier, Abdalla & Joachimi (2016), MNRAS 459, 3693 — lognormal mocks (FLASK)
  [arXiv:1602.08503](https://arxiv.org/abs/1602.08503).
- Desjacques, Jeong & Schmidt (2018), Phys. Rep. 733, 1 — galaxy bias review
  [arXiv:1611.09787](https://arxiv.org/abs/1611.09787).
- Bardeen, Bond, Kaiser & Szalay (1986), ApJ 304, 15 — transfer function
  [doi:10.1086/164143](https://doi.org/10.1086/164143).
- **Tessore et al. (2023)**, Open J. Astrophys. — GLASS
  [arXiv:2302.01942](https://arxiv.org/abs/2302.01942).


## 1. A Gaussian random field from $P(k)$

A Gaussian field is fully specified by $P(k)$. The recipe is: draw white noise, Fourier transform,
colour each mode by $\sqrt{P(k)}$, and transform back (Peebles 1980):

$$\delta_G(\mathbf{x}) = \mathcal{F}^{-1}\!\left[\sqrt{P(k)}\;\mathcal{F}[n](\mathbf{k})\right],
\quad n(\mathbf{x})\sim\mathcal{N}(0,1).$$

We use the BBKS (1986) shape $P(k)\propto k^{n_s}T^2(k)$ and a JAX FFT so the whole construction is
differentiable.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp

plt.rcParams.update({'figure.figsize': (10, 6), 'font.size': 11})

h, ns = 0.6736, 0.9649
N, Lbox = 256, 500.0                       # grid points, box size [Mpc/h]
kf = 2*np.pi/Lbox
kx = jnp.fft.fftfreq(N, d=Lbox/N) * 2*jnp.pi      # h/Mpc
KX, KY = jnp.meshgrid(kx, kx, indexing='ij')
Kmag = jnp.sqrt(KX**2 + KY**2)

def T_bbks(k, G):
    q = jnp.where(k > 0, k, 1e-12)/G
    return jnp.log(1+2.34*q)/(2.34*q)*(1+3.89*q+(16.1*q)**2+(5.46*q)**3+(6.71*q)**4)**-0.25

def Pk2d(k, Om=0.31):
    return jnp.where(k > 0, k**ns * T_bbks(k, Om*h)**2, 0.0)

def gaussian_field(key, sigma_g, Om=0.31):
    wn = jax.random.normal(key, (N, N))
    fk = jnp.fft.fft2(wn) * jnp.sqrt(Pk2d(Kmag, Om))
    g = jnp.fft.ifft2(fk).real
    return g / jnp.std(g) * sigma_g           # scale to target Gaussian rms

key = jax.random.key(7)
sigma_g = 0.9
g = gaussian_field(key, sigma_g)
plt.figure(figsize=(6, 5))
plt.imshow(np.array(g), cmap='RdBu_r', extent=[0, Lbox, 0, Lbox])
plt.colorbar(label=r'$\delta_G$'); plt.title('Gaussian field'); plt.xlabel('Mpc/h'); plt.show()


## 2. Lognormal transform and galaxy bias

A Gaussian field allows $\delta<-1$ (negative density). The **lognormal** field fixes this and matches
the observed skewed one-point distribution (Coles & Jones 1991):

$$\delta_{\rm LN} = \exp\!\left(\delta_G - \tfrac12\sigma_G^2\right) - 1,$$

with mean zero by construction and variance $\langle\delta_{\rm LN}^2\rangle=e^{\sigma_G^2}-1$. Galaxies
trace mass with a **linear bias** $b$ (Desjacques et al. 2018), giving an expected count
$\lambda = \bar n\,(1 + b\,\delta_{\rm LN})$, which we Poisson-sample.


In [ ]:
def lognormal_field(g, sigma_g):
    return jnp.exp(g - 0.5*sigma_g**2) - 1.0

delta_LN = lognormal_field(g, sigma_g)
bias, nbar = 1.8, 40.0                         # bias, mean galaxies per cell
lam = np.clip(np.array(nbar*(1 + bias*delta_LN)), 0, None)
counts = np.random.default_rng(1).poisson(lam)

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
im0 = ax[0].imshow(np.array(delta_LN), cmap='RdBu_r', extent=[0, Lbox, 0, Lbox])
ax[0].set_title(r'lognormal $\delta_{\rm LN}$'); fig.colorbar(im0, ax=ax[0])
im1 = ax[1].imshow(counts, cmap='inferno', extent=[0, Lbox, 0, Lbox])
ax[1].set_title('Poisson galaxy counts'); fig.colorbar(im1, ax=ax[1])
for a in ax: a.set_xlabel('Mpc/h')
fig.savefig('density_field_galaxy_map.png', dpi=130); plt.show()
print(f"theory var(delta_LN)=exp(sig^2)-1={np.exp(sigma_g**2)-1:.3f}, measured={np.var(np.array(delta_LN)):.3f}")


## 3. Recover $P(k)$ from the mock

The acid test of a mock is that its measured power spectrum matches the input. We FFT the galaxy
overdensity $\delta_g = n/\bar n - 1$, square, and bin in $|k|$, subtracting the Poisson shot noise
$1/\bar n$.


In [ ]:
def measure_pk(field2d):
    fk = np.fft.fft2(field2d)
    p2d = (np.abs(fk)**2) * (Lbox**2) / N**4
    kk = np.array(Kmag)
    kbins = np.linspace(kf, kx.max().item()*0.7, 25)
    kc = 0.5*(kbins[1:]+kbins[:-1])
    pk = np.array([p2d[(kk >= kbins[i]) & (kk < kbins[i+1])].mean() for i in range(len(kc))])
    return kc, pk

dg = counts/counts.mean() - 1.0
kc, pk_meas = measure_pk(dg)
pk_meas_shot = pk_meas - (Lbox**2/counts.sum())          # subtract shot noise
kc_m, pk_in = measure_pk(np.array(bias*delta_LN))

plt.figure()
plt.loglog(kc, pk_meas_shot, 'o', label='galaxies (shot-noise subtracted)')
plt.loglog(kc_m, pk_in, '-', label=r'input $b\,\delta_{\rm LN}$')
plt.xlabel('k [h/Mpc]'); plt.ylabel('P(k) [arb.]'); plt.legend()
plt.title('Recovered vs input power spectrum'); plt.show()


## 4. The map-making is differentiable

Crucially for the thesis, the entire Gaussian$\to$lognormal forward map is differentiable. The
*ensemble* variance of $\delta_{\rm LN}$ has the closed form $e^{\sigma_G^2}-1$ (Coles & Jones 1991),
with derivative $2\sigma_G e^{\sigma_G^2}$. Autodiff through the FFT pipeline differentiates the *single
realisation* we generated, so it tracks the analytic ensemble derivative up to the realisation's sample
scatter — exactly the gradient a field-level sampler (N13) would back-propagate through the map.


In [ ]:
def lognormal_variance(sig):
    field = lognormal_field(gaussian_field(key, sig), sig)
    return jnp.var(field)

g_auto = float(jax.grad(lognormal_variance)(0.9))
g_theory = 2*0.9*np.exp(0.9**2)
print(f"d var(delta_LN)/d sigma_G : autodiff={g_auto:.3f}  theory 2*sig*exp(sig^2)={g_theory:.3f}")


## 5. A realistic mock on the sphere with GLASS

For survey work the field must live on the sphere across redshift shells. **GLASS** (Tessore et al.
2023) generates correlated **lognormal** matter shells from the angular power spectra and Poisson-samples
galaxy positions in each shell. Here we generate one lognormal shell at $z\simeq0.5$ (its angular
$C_\ell$ from CCL) and sample galaxies from it — the spherical analogue of Sections 1–3.


In [ ]:
import glass, glass.grf, healpy as hp, pyccl as ccl

nside_g = 128
cosmo_g = ccl.Cosmology(Omega_c=0.264, Omega_b=0.049, h=0.6736, n_s=0.9649, sigma8=0.81)
zsh = np.linspace(0.01, 1.2, 120)
nsh = np.exp(-0.5*((zsh - 0.5)/0.15)**2)                          # matter shell around z ~ 0.5
mat_tr = ccl.NumberCountsTracer(cosmo_g, has_rsd=False, dndz=(zsh, nsh), bias=(zsh, np.ones_like(zsh)))
lmax_g = 2*nside_g - 1
cl_shell = np.concatenate([[0.0, 0.0],
                           ccl.angular_cl(cosmo_g, mat_tr, mat_tr, np.arange(2, lmax_g+1).astype(float))])

fields = [glass.grf.Lognormal()]
with np.errstate(divide='ignore', invalid='ignore'):
    gls = glass.solve_gaussian_spectra(fields, [cl_shell])
gls = [np.clip(np.nan_to_num(g), 0.0, None) for g in gls]
delta_sphere = next(iter(glass.generate(fields, gls, nside_g)))

lon, lat, _ = next(iter(glass.positions_from_delta(30.0, delta_sphere, 1.8)))   # galaxy bias 1.8
print("GLASS lognormal shell: std=%.3f min=%.3f  ->  %d galaxies sampled"
      % (delta_sphere.std(), delta_sphere.min(), np.size(lon)))

hp.mollview(delta_sphere, title=r'GLASS lognormal matter shell on the sphere ($z\simeq0.5$)',
            cmap='RdBu_r', min=-0.4, max=0.4)
plt.savefig('density_glass_sphere.png', dpi=130); plt.show()


## Exercises

1. **Gaussian vs lognormal one-point.** Histogram $\delta_G$ and $\delta_{\rm LN}$; show the lognormal is
   skewed and bounded below by $-1$.
2. **Bias and clustering.** Vary $b\in\{1,2,3\}$ and verify the recovered $P(k)$ amplitude scales as $b^2$.
3. **Shot noise.** Lower $\bar n$ and show the shot-noise plateau $1/\bar n$ rises, swamping small scales.
4. **Differentiable summary.** Differentiate the measured large-scale $P(k)$ amplitude with respect to the
   bias $b$ by autodiff and compare to the expected $2b$ scaling.


## Summary

- A Gaussian field follows from $P(k)$ via an FFT; the lognormal transform makes it physical.
- Linear bias + Poisson sampling turns the matter field into a galaxy map; the input $P(k)$ is recovered.
- The forward map is differentiable end-to-end (autodiff matches the Coles & Jones variance) — the basis
  of field-level inference in N13.
- GLASS extends this to correlated lognormal shells on the sphere.

**Next (N09):** lens that matter field to get weak-lensing convergence and shear.
